# ASVspoof 2019 Physical Access — Complete Reproducibility Workflow

This notebook orchestrates the existing project implementation from dataset discovery through feature extraction, E1–E7 ablation, tuned E7 training/evaluation, explainability, condition analysis, and paper figure/table generation.

**It calls the project's existing Python modules and scripts. It does not reimplement the feature extractors or model.** Run `ASVspoof_2019_PA_Pipeline_Smoke_Test.ipynb` first. After the smoke test passes, run this notebook from the project environment, preferably with the notebook kernel using the project's `.venv`.

## Workflow order

1. Configure the dataset and ASV-score paths; inspect the expected dataset layout.
2. Check project files, environment, GPU support and run unit tests.
3. Generate protocol manifests; optionally run a complete audio-decode validation.
4. Extract CQCC, TECC/TKEO and RLFCC/acoustic features; aggregate per-utterance statistics; fuse matrices.
5. Train E1–E7 baseline/ablation models and calculate official PA min-tDCF.
6. Tune E7 using the official training and development splits only; refit the selected model on all training data; evaluate once on the official evaluation split.
7. Generate global, class-conditional and four individual SHAP explanations.
8. Run the E1–E7 benchmark-condition analysis and paired statistical comparisons.
9. Generate the paper figures and CSV tables.

> **Evaluation discipline:** `tools/tune_e7.py` must not read evaluation data for model selection. It uses the official development split for early stopping and hyperparameter ranking, then refits on all official training data. The untouched evaluation split is processed by `tools/evaluate_tuned_e7.py` after tuning. Do not change the tuner to inspect evaluation metrics while selecting trials.

> **Compute and storage:** full extraction and 80-trial XGBoost tuning are substantial jobs. Extraction is CPU-based in the current implementation; XGBoost can use CUDA. The frame-level cache can be large. The default workflow aggregates each group/split and then deletes that group/split's frame cache only after the matrix has been written and checked. Turn that cleanup off if you need to retain frame caches.

## 1. Dataset layout expected by the current project configuration

Edit `DATASET_ROOT` in the next cell. It must point to the **parent directory containing `PA`**, not directly to `PA`.

The current `configs/data.yaml` specifies these names:

```text
<DATASET_ROOT>/
└── PA/
    ├── ASVspoof2019_PA_train/
    │   └── flac/                         # training .flac utterances
    ├── ASVspoof2019_PA_dev/
    │   └── flac/                         # development .flac utterances
    ├── ASVspoof2019_PA_eval/
    │   └── flac/                         # evaluation .flac utterances
    ├── ASVspoof2019_PA_cm_protocols/
    │   ├── ASVspoof2019.PA.cm.train.trn.txt
    │   ├── ASVspoof2019.PA.cm.dev.trl.txt
    │   └── ASVspoof2019.PA.cm.eval.trl.txt
    └── ASVspoof2019_PA_asv_scores/
        ├── ASVspoof2019.PA.asv.dev.gi.trl.scores.txt
        └── ASVspoof2019.PA.asv.eval.gi.trl.scores.txt
```

The `ASVspoof2019_PA_asv_scores` directory is used by the official tandem detection cost calculation and E7 tuning/evaluation. If your ASV scores are stored elsewhere, edit `ASV_SCORE_DIR` separately. File and directory names are case-sensitive on Linux.

The CM protocol is authoritative for labels and trial membership. The code expects five whitespace-separated fields per non-empty protocol line: `speaker_id utterance_id environment attack_id key`, with `key` equal to `bonafide` or `spoof`.

In [ ]:
import os
from pathlib import Path

# Set ASVSPOOF_DATASET_ROOT in the environment, or edit this path.
# It must point to the parent directory containing the PA folder.
DATASET_ROOT = Path(os.environ.get("ASVSPOOF_DATASET_ROOT", "/path/to/ASVspoof2019_archive"))

# Optional environment overrides. ASVSPOOF_ASV_SCORE_DIR should point directly
# to ASVspoof2019_PA_asv_scores when the score files are stored separately.
ASV_SCORE_DIR_OVERRIDE = os.environ.get("ASVSPOOF_ASV_SCORE_DIR") or None
PROJECT_ROOT_OVERRIDE = os.environ.get("ASVSPOOF_PROJECT_ROOT") or None

# Compute settings. Feature extraction in the current code uses CPU workers.
EXTRACTION_WORKERS = 6
XGB_N_JOBS = 6
XGB_DEVICE = "cuda"  # set to "cpu" on a machine without CUDA-enabled XGBoost

# Stage switches. Keep True to run the complete pipeline in order.
INSTALL_PROJECT_EXTRAS = False  # True installs the project and [dev,explain] extras
RUN_UNIT_TESTS = True
RUN_FULL_AUDIO_VALIDATION = False  # True performs a full decode/read scan of all audio files
RUN_FEATURE_PIPELINE = True
RUN_E1_E7_ABLATIONS = True
RUN_ABLATION_TDCF = True
RUN_E7_FINAL_TUNING = True
RUN_E7_FINAL_EVALUATION = True
RUN_FINAL_SHAP = True
RUN_E1_E7_CONDITION_ANALYSIS = True
RUN_PAIRED_CONDITION_STATISTICS = True
RUN_PAPER_FIGURES_AND_TABLES = True

# Safe-resume/rebuild controls.
SKIP_COMPLETED_GROUP_MATRICES = True
REBUILD_GROUP_MATRICES = False
REBUILD_FUSION_MATRICES = False
DELETE_FRAME_CACHES_AFTER_AGGREGATION = True
SKIP_COMPLETED_ABLATIONS = True
FORCE_NEW_TUNING_SEARCH = False  # set True only to discard the previous search and start over

# Experiment configuration, matching the existing scripts and saved runs.
AGGREGATION_STATS = ["mean", "std", "p10", "p50", "p90"]
ABLATION_N_ESTIMATORS = 2000
ABLATION_EARLY_STOPPING = 100
TUNING_TRIALS = 80
TUNING_MAX_ESTIMATORS = 10000
TUNING_EARLY_STOPPING = 200
RANDOM_SEED = 42
SHAP_GLOBAL_SAMPLE_SIZE = 2000
SHAP_CLASS_SAMPLES_PER_CLASS = 1000
SHAP_BATCH_SIZE = 256

# Evaluation cases used for the paper's individual SHAP figures.
INDIVIDUAL_CASES = [
    ("PA_E_0047911", "false_accept"),
    ("PA_E_0110936", "false_reject"),
    ("PA_E_0076063", "correct_accept"),
    ("PA_E_0023389", "correct_reject"),
]

## 2. Project structure and dependency setup

The repository must include the complete Python source tree under `src/`, the command-line entry point `src/cli.py`, the research scripts under `tools/`, and the project configuration files. In particular, retain the feature and data modules under `src/features/` and `src/data/`; these are implementation source files, not generated dataset artifacts.

Generated audio caches, intermediate feature matrices, trained models, logs and result files are runtime outputs. They can be regenerated by this workflow and should normally remain outside the source-only GitHub release.

In [ ]:
import os
import sys
import json
import time
import shutil
import subprocess
import platform
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "cli.py").is_file() and (candidate / "tools" / "tune_e7.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Set PROJECT_ROOT_OVERRIDE to the directory "
        "containing src/cli.py and tools/tune_e7.py."
    )

PROJECT_ROOT = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE is not None
    else find_project_root(Path.cwd())
)
if not PROJECT_ROOT.is_dir():
    raise NotADirectoryError(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

PA_ROOT = DATASET_ROOT.expanduser().resolve() / "PA"
ASV_SCORE_DIR = (
    Path(ASV_SCORE_DIR_OVERRIDE).expanduser().resolve()
    if ASV_SCORE_DIR_OVERRIDE is not None
    else PA_ROOT / "ASVspoof2019_PA_asv_scores"
)

CONFIG_SOURCE = PROJECT_ROOT / "configs" / "data.yaml"
LOCAL_CONFIG = PROJECT_ROOT / "configs" / "data.notebook.local.yaml"

required_project_files = [
    "requirements.txt", "pyproject.toml", "configs/data.yaml", "src/cli.py", "src/config.py",
    "src/data/audio.py", "src/data/manifest.py", "src/data/protocol.py", "src/data/validation.py",
    "src/features/cqt.py", "src/features/tkeo.py", "src/features/acoustic.py",
    "src/pipeline/cqcc.py", "src/pipeline/tkeo.py", "src/pipeline/acoustic.py",
    "src/pipeline/fusion.py", "src/pipeline/cache.py", "src/models/xgb_fusion.py",
    "tools/tune_e7.py", "tools/evaluate_tuned_e7.py", "tools/compute_pa_tdcf.py",
    "tools/integrate_tdcf_results.py", "tools/shap_global_e7.py",
    "tools/shap_class_conditional_e7.py", "tools/explain_one_trial.py",
    "tools/ASVspoof_PA_condition_analysis.py", "tools/paired_pa_condition_stats.py",
    "tools/generate_e7_paper_figures.py", "tools/tdcf/eval_metrics.py",
]
missing_project_files = [f for f in required_project_files if not (PROJECT_ROOT / f).is_file()]
if missing_project_files:
    raise FileNotFoundError(
        "The project source tree is incomplete. Missing required files:\n - "
        + "\n - ".join(missing_project_files)
        + "\n\nRestore the missing source files from the project repository before continuing."
    )

if not CONFIG_SOURCE.is_file():
    raise FileNotFoundError(CONFIG_SOURCE)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("PA directory:", PA_ROOT)
print("ASV score directory:", ASV_SCORE_DIR)
print("Notebook kernel:", sys.executable)
print("Python:", sys.version.split()[0])

In [ ]:
# Optional: install project dependencies into the active notebook kernel.
if INSTALL_PROJECT_EXTRAS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev,explain]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Installed project dependencies and explainability extras.")
else:
    print("Dependency installation skipped. Set INSTALL_PROJECT_EXTRAS=True if needed.")

import yaml
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import scipy
from IPython.display import display

try:
    import shap
    SHAP_VERSION = shap.__version__
except ImportError:
    SHAP_VERSION = None

try:
    import matplotlib
    MATPLOTLIB_VERSION = matplotlib.__version__
except ImportError:
    MATPLOTLIB_VERSION = None

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("XGBoost build:", xgb.build_info())
print("SHAP:", SHAP_VERSION)
print("Matplotlib:", MATPLOTLIB_VERSION)

if XGB_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'.")
if XGB_DEVICE == "cuda" and xgb.build_info().get("USE_CUDA") is False:
    raise RuntimeError(
        "XGB_DEVICE='cuda' but this XGBoost build reports USE_CUDA=False. "
        "Install a CUDA-enabled XGBoost build or set XGB_DEVICE='cpu'."
    )

## 3. Create a local dataset configuration without modifying the shared template

The original `configs/data.yaml` is retained unchanged. This cell writes `configs/data.notebook.local.yaml` with the configured dataset root and uses that local file for all commands. The local file contains an absolute path specific to the machine running the notebook; the repository `.gitignore` should exclude it.

In [ ]:
base_config = yaml.safe_load(CONFIG_SOURCE.read_text(encoding="utf-8"))
base_config["data"]["root"] = str(DATASET_ROOT.expanduser().resolve())
base_config["data"]["access_dir"] = "PA"
LOCAL_CONFIG.parent.mkdir(parents=True, exist_ok=True)
LOCAL_CONFIG.write_text(yaml.safe_dump(base_config, sort_keys=False), encoding="utf-8")
CONFIG_ARG = str(LOCAL_CONFIG.relative_to(PROJECT_ROOT))

# Required paths are derived from the same data.yaml that the CLI uses.
required_dataset_paths = {}
for split in ("train", "dev", "eval"):
    required_dataset_paths[f"{split} protocol"] = PA_ROOT / base_config["data"]["protocols"][split]
    required_dataset_paths[f"{split} audio directory"] = PA_ROOT / base_config["data"]["audio_dirs"][split]

missing_dataset_paths = [f"{label}: {path}" for label, path in required_dataset_paths.items() if not path.exists()]
if missing_dataset_paths:
    raise FileNotFoundError(
        "Dataset layout does not match configs/data.yaml. Missing:\n - "
        + "\n - ".join(missing_dataset_paths)
        + "\nEdit DATASET_ROOT or update the protocol/audio directory entries in configs/data.yaml."
    )

asv_required = [
    ASV_SCORE_DIR / "ASVspoof2019.PA.asv.dev.gi.trl.scores.txt",
    ASV_SCORE_DIR / "ASVspoof2019.PA.asv.eval.gi.trl.scores.txt",
]
if RUN_ABLATION_TDCF or RUN_E7_FINAL_TUNING or RUN_E7_FINAL_EVALUATION:
    missing_asv = [str(p) for p in asv_required if not p.is_file()]
    if missing_asv:
        raise FileNotFoundError(
            "Required official ASV score files were not found:\n - " + "\n - ".join(missing_asv)
            + "\nEdit ASV_SCORE_DIR_OVERRIDE if the score directory is stored elsewhere."
        )

print("Runtime configuration:", LOCAL_CONFIG)
for label, path in required_dataset_paths.items():
    if path.is_dir():
        print(f"{label:18s}: {path} | FLAC files: {len(list(path.glob('*.flac'))):,}" if "audio" in label else f"{label:18s}: {path}")
    else:
        print(f"{label:18s}: {path}")
print("ASV dev scores:", asv_required[0].exists(), asv_required[0])
print("ASV eval scores:", asv_required[1].exists(), asv_required[1])

## 4. Helper functions and validation

`run_cli` calls the repository's CLI with the generated local configuration. `run_tool` invokes the existing research scripts using the current notebook interpreter. All processes run with the project root as their working directory, so relative paths remain consistent.

In [ ]:
def run_cli(*args, check=True):
    cmd = [sys.executable, "-m", "src.cli", *map(str, args)]
    print("\n$", " ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=check)


def run_tool(script, *args, check=True):
    script_path = PROJECT_ROOT / script
    if not script_path.is_file():
        raise FileNotFoundError(script_path)
    cmd = [sys.executable, str(script_path), *map(str, args)]
    print("\n$", " ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=check)


def relative(path):
    return str(Path(path).relative_to(PROJECT_ROOT)) if Path(path).is_absolute() else str(path)


def require_paths(paths, label):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(label + " is missing required files:\n - " + "\n - ".join(missing))


PROCESSED_DIR = PROJECT_ROOT / base_config["output"]["processed_dir"]
MANIFEST_DIR = PROJECT_ROOT / base_config["output"]["manifest_dir"]
CACHE_ROOT = PROJECT_ROOT / base_config["output"]["cache_dir"]
GROUP_DIR = PROJECT_ROOT / "results" / "group_features"
FUSION_DIR = PROJECT_ROOT / "results" / "fusion_features"
ABLATION_DIR = PROJECT_ROOT / "results" / "xgb_ablation"
TDCF_DIR = PROJECT_ROOT / "results" / "tdcf"
TUNED_DIR = PROJECT_ROOT / "results" / "xgb_tuning" / "E7_FINAL"
FINAL_MODEL_DIR = TUNED_DIR / "best_model"
GLOBAL_SHAP_DIR = PROJECT_ROOT / "results" / "explainability" / "E7_FINAL" / "global"
CLASS_SHAP_DIR = PROJECT_ROOT / "results" / "explainability" / "E7_FINAL" / "class_conditional"
INDIVIDUAL_SHAP_DIR = PROJECT_ROOT / "results" / "explainability" / "E7_FINAL" / "individual"
PAPER_FIGURE_DIR = PROJECT_ROOT / "results" / "paper_figures" / "E7_FINAL"

## 5. Unit tests and protocol-derived manifests

The manifest builder reads labels and trial metadata from the official CM protocol files. It also checks that utterance IDs do not overlap across the selected splits. This stage does not decode the audio.

In [ ]:
if RUN_UNIT_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_ROOT, check=True)
else:
    print("Unit tests skipped.")

run_cli("manifest", "--config", CONFIG_ARG, "--splits", "train", "dev", "eval")

manifest_summary = {}
missing_audio = {}
for split in ("train", "dev", "eval"):
    manifest_path = MANIFEST_DIR / f"PA_{split}.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    df = pd.read_csv(manifest_path, dtype={"utterance_id": str})
    absent = [p for p in df["audio_path"].astype(str) if not Path(p).is_file()]
    manifest_summary[split] = {
        "utterances": int(len(df)),
        "bonafide": int((df["key"] == "bonafide").sum()),
        "spoof": int((df["key"] == "spoof").sum()),
        "missing_audio": len(absent),
        "manifest": str(manifest_path.relative_to(PROJECT_ROOT)),
    }
    missing_audio[split] = absent[:10]

pd.DataFrame(manifest_summary).T

In [ ]:
for split, summary in manifest_summary.items():
    print(split.upper(), summary)
    if summary["missing_audio"]:
        print("Missing audio examples:", missing_audio[split])
        raise FileNotFoundError(f"{summary['missing_audio']} audio paths are missing for {split}.")

if RUN_FULL_AUDIO_VALIDATION:
    # This scans/decodes every utterance in each selected split and can take a long time.
    run_cli("validate", "--config", CONFIG_ARG, "--splits", "train", "dev", "eval")
else:
    print("Full audio decode validation skipped. Manifest/audio-path checks passed.")

## 6. Frame-level feature extraction and utterance-level aggregation

The three frame-level feature groups are:

- **CQCC:** 90 coefficients per frame; five summary statistics produce 450 utterance-level features.
- **TECC/TKEO:** 120 coefficients per frame; five summary statistics produce 600 features.
- **RLFCC/acoustic:** 60 coefficients per frame; five summary statistics produce 300 features.

The combined E7 representation therefore contains 1,350 features. The feature extractor parameters remain in the project configuration and feature-code modules; this notebook does not silently override them.

Each split/group is processed separately. The code reuses valid cached utterance features. After an aggregate matrix is written and row-count checked, the corresponding frame cache can be removed to save storage.

In [ ]:
FEATURE_GROUPS = ["cqcc", "tkeo", "acoustic"]
EXTRACT_COMMANDS = {"cqcc": "extract-cqcc", "tkeo": "extract-tkeo", "acoustic": "extract-acoustic"}
EXPECTED_GROUP_WIDTHS = {"cqcc": 450, "tkeo": 600, "acoustic": 300}

if RUN_FEATURE_PIPELINE:
    for split in ("train", "dev", "eval"):
        manifest_path = MANIFEST_DIR / f"PA_{split}.csv"
        n_rows_expected = len(pd.read_csv(manifest_path))
        for group in FEATURE_GROUPS:
            group_dir = GROUP_DIR / group
            x_path = group_dir / f"X_{split}.npy"
            y_path = group_dir / f"y_{split}.npy"
            meta_path = group_dir / f"metadata_{split}.csv"
            matrix_parts = (group_dir / "feature_schema.json", x_path, y_path, meta_path)
            matrix_exists = all(p.is_file() for p in matrix_parts)
            partial_matrix = any(p.exists() for p in matrix_parts) and not matrix_exists

            if matrix_exists and SKIP_COMPLETED_GROUP_MATRICES and not REBUILD_GROUP_MATRICES:
                X_existing = np.load(x_path, mmap_mode="r")
                if X_existing.shape != (n_rows_expected, EXPECTED_GROUP_WIDTHS[group]):
                    raise ValueError(f"Existing {group}/{split} matrix has unexpected shape {X_existing.shape}.")
                print(f"SKIP existing aggregate {group}/{split}: {X_existing.shape}")
                del X_existing
                continue

            extract_args = [
                EXTRACT_COMMANDS[group], "--config", CONFIG_ARG,
                "--splits", split, "--workers", EXTRACTION_WORKERS,
                "--cache-root", relative(CACHE_ROOT),
            ]
            if REBUILD_GROUP_MATRICES:
                extract_args.append("--overwrite")
            run_cli(*extract_args)

            report_path = PROCESSED_DIR / "feature_reports" / f"{group}_{split}_summary.json"
            if not report_path.is_file():
                raise FileNotFoundError(report_path)
            report = json.loads(report_path.read_text(encoding="utf-8"))
            if report.get("failed_count", 0) != 0:
                raise RuntimeError(f"{group}/{split} extraction had failures. See {report_path}")
            if report.get("written", 0) + report.get("skipped_cached", 0) != report.get("total_requested"):
                raise RuntimeError(f"{group}/{split}: extraction completion counts do not match report total.")

            matrix_args = [
                "build-group-matrix", "--config", CONFIG_ARG,
                "--splits", split, "--feature-group", group,
                "--cache-root", relative(CACHE_ROOT),
                "--output-dir", relative(GROUP_DIR), "--stats", *AGGREGATION_STATS,
            ]
            if REBUILD_GROUP_MATRICES or partial_matrix:
                matrix_args.append("--overwrite")
            run_cli(*matrix_args)

            X_group = np.load(x_path, mmap_mode="r")
            y_group = np.load(y_path, mmap_mode="r")
            if X_group.shape != (n_rows_expected, EXPECTED_GROUP_WIDTHS[group]):
                raise ValueError(f"{group}/{split} aggregate shape is {X_group.shape}, expected {(n_rows_expected, EXPECTED_GROUP_WIDTHS[group])}.")
            if len(y_group) != n_rows_expected:
                raise ValueError(f"{group}/{split} label count does not match manifest rows.")
            del X_group, y_group

            if DELETE_FRAME_CACHES_AFTER_AGGREGATION:
                cache_split = CACHE_ROOT / "features" / group / split
                if cache_split.exists():
                    shutil.rmtree(cache_split)
                    print("Removed validated frame cache:", cache_split)
else:
    print("Feature extraction and aggregation skipped by RUN_FEATURE_PIPELINE=False.")

## 7. Combine the three feature groups into the fused train/dev/eval matrices

The fusion stage checks that the group matrices have identical utterance ordering and labels before concatenating them. E7 must contain exactly 1,350 dimensions.

In [ ]:
if RUN_FEATURE_PIPELINE:
    fused_required = [
        FUSION_DIR / f"{prefix}_{split}.npy"
        for split in ("train", "dev", "eval")
        for prefix in ("X", "y")
    ] + [FUSION_DIR / "feature_schema.json"]
    fusion_is_complete = all(p.is_file() for p in fused_required)
    if REBUILD_FUSION_MATRICES or not fusion_is_complete:
        combine_args = [
            "combine-fusion", "--config", CONFIG_ARG,
            "--splits", "train", "dev", "eval",
            "--feature-groups", *FEATURE_GROUPS,
            "--group-feature-dir", relative(GROUP_DIR),
            "--output-dir", relative(FUSION_DIR),
        ]
        if REBUILD_FUSION_MATRICES:
            combine_args.append("--overwrite")
        run_cli(*combine_args)
    else:
        print("All fused train/dev/eval matrices already exist; reusing them.")

    fusion_schema = json.loads((FUSION_DIR / "feature_schema.json").read_text(encoding="utf-8"))
    if int(fusion_schema.get("n_features", -1)) != 1350:
        raise ValueError(f"Expected 1,350 fused features, found {fusion_schema.get('n_features')}")
    fusion_shapes = {}
    for split in ("train", "dev", "eval"):
        X = np.load(FUSION_DIR / f"X_{split}.npy", mmap_mode="r")
        y = np.load(FUSION_DIR / f"y_{split}.npy", mmap_mode="r")
        metadata = pd.read_csv(FUSION_DIR / f"metadata_{split}.csv")
        if X.shape != (len(metadata), 1350) or len(y) != len(metadata):
            raise ValueError(f"Fused matrix/label/metadata alignment problem for {split}: {X.shape}, {len(y)}, {len(metadata)}")
        fusion_shapes[split] = X.shape
        del X, y
    print("Fused feature dimensions:", fusion_shapes)
else:
    print("Fusion step skipped. Existing results/fusion_features must be present before model stages.")

## 8. Train the E1–E7 feature-ablation models

The seven combinations are defined in the project workflow:

| Experiment | Feature groups | Feature count |
|---|---|---:|
| E1 | CQCC | 450 |
| E2 | TECC/TKEO | 600 |
| E3 | RLFCC/acoustic | 300 |
| E4 | CQCC + TECC/TKEO | 1,050 |
| E5 | CQCC + RLFCC/acoustic | 750 |
| E6 | TECC/TKEO + RLFCC/acoustic | 900 |
| E7 | All three groups | 1,350 |

These baseline/ablation runs use the existing `train-xgb` implementation and its fixed experiment settings. Each run fits on train, early-stops/chooses its threshold on dev, and writes evaluation predictions and metrics. The later E7_FINAL tuning script is a separate experiment.

In [ ]:
ABLATIONS = {
    "E1_CQCC": ["cqcc"],
    "E2_TKEO": ["tkeo"],
    "E3_ACOUSTIC": ["acoustic"],
    "E4_CQCC_TKEO": ["cqcc", "tkeo"],
    "E5_CQCC_ACOUSTIC": ["cqcc", "acoustic"],
    "E6_TKEO_ACOUSTIC": ["tkeo", "acoustic"],
    "E7_ALL": ["cqcc", "tkeo", "acoustic"],
}

if RUN_E1_E7_ABLATIONS:
    for experiment, groups in ABLATIONS.items():
        model_dir = ABLATION_DIR / experiment
        required_outputs = [model_dir / "metrics.json", model_dir / "xgb_fusion.json", model_dir / "eval_predictions.csv"]
        if SKIP_COMPLETED_ABLATIONS and all(p.is_file() for p in required_outputs):
            print("SKIP completed ablation:", experiment)
            continue
        args = [
            "train-xgb", "--config", CONFIG_ARG,
            "--feature-groups", *groups,
            "--feature-dir", relative(FUSION_DIR),
            "--model-dir", relative(model_dir),
            "--n-estimators", ABLATION_N_ESTIMATORS,
            "--early-stopping-rounds", ABLATION_EARLY_STOPPING,
            "--max-depth", 5, "--learning-rate", 0.03,
            "--n-jobs", XGB_N_JOBS, "--device", XGB_DEVICE,
        ]
        run_cli(*args)
else:
    print("E1–E7 ablation training skipped.")

## 9. Official PA min-tDCF for E1–E7

The official ASV score files are required in addition to the CM scores. The first script computes dev and eval min-tDCF using the bundled `tools/tdcf/eval_metrics.py`; the second merges those values into each ablation `metrics.json` and rebuilds `results/tdcf/ablation_eer_tdcf_comparison.csv` for the paper-figure generator.

In [ ]:
if RUN_ABLATION_TDCF:
    run_tool(
        "tools/compute_pa_tdcf.py",
        "--project-root", str(PROJECT_ROOT),
        "--asv-dir", str(ASV_SCORE_DIR),
        "--output-dir", str(TDCF_DIR),
    )
    run_tool(
        "tools/integrate_tdcf_results.py",
        "--project-root", str(PROJECT_ROOT),
        "--tdcf-csv", str(TDCF_DIR / "pa_min_tdcf_summary.csv"),
    )
    ablation_comparison = TDCF_DIR / "ablation_eer_tdcf_comparison.csv"
    if not ablation_comparison.is_file():
        raise FileNotFoundError(ablation_comparison)
    display(pd.read_csv(ablation_comparison))
else:
    print("E1–E7 min-tDCF calculation skipped.")

## 10. Tune E7_FINAL using train and dev only

This stage uses the existing 80-trial search script. It reads the fused train and dev matrices plus the development ASV scores. It does **not** load evaluation matrices, labels, metadata or predictions. Hyperparameters are ranked by dev min-tDCF (ties: lower dev EER, then higher dev ROC-AUC). The selected configuration is refit on all official training data using the winning trial's selected tree count.

The tuner can resume completed trials when the saved `search_manifest.json` matches the current settings. Do not set `FORCE_NEW_TUNING_SEARCH=True` unless you intentionally want to discard the previous trial log and start a fresh search.

In [ ]:
if RUN_E7_FINAL_TUNING:
    tuning_args = [
        "--feature-dir", str(FUSION_DIR),
        "--asv-dir", str(ASV_SCORE_DIR),
        "--output-dir", str(TUNED_DIR),
        "--trials", TUNING_TRIALS,
        "--seed", RANDOM_SEED,
        "--n-estimators", TUNING_MAX_ESTIMATORS,
        "--early-stopping-rounds", TUNING_EARLY_STOPPING,
        "--n-jobs", XGB_N_JOBS,
        "--device", XGB_DEVICE,
        "--verbose", 100,
    ]
    if FORCE_NEW_TUNING_SEARCH:
        tuning_args.append("--force")
    run_tool("tools/tune_e7.py", *tuning_args)
else:
    print("E7_FINAL tuning skipped; a completed results/xgb_tuning/E7_FINAL model must already exist.")

require_paths([FINAL_MODEL_DIR / "xgb_fusion.json", FINAL_MODEL_DIR / "metrics.json", FINAL_MODEL_DIR / "feature_names.json"], "E7_FINAL model")

## 11. One-pass evaluation of E7_FINAL on the official evaluation split

This script loads the saved tuned model and the evaluation matrix. It applies the fixed threshold selected on development data for operational FAR/FRR/confusion-matrix metrics, and separately calculates threshold-independent EER/ROC-AUC and official min-tDCF. It also writes retrospective FAR summaries by attack and environment metadata; those metadata are not model inputs.

In [ ]:
if RUN_E7_FINAL_EVALUATION:
    run_tool(
        "tools/evaluate_tuned_e7.py",
        "--project-root", str(PROJECT_ROOT),
        "--feature-dir", str(FUSION_DIR),
        "--model-dir", str(FINAL_MODEL_DIR),
        "--asv-dir", str(ASV_SCORE_DIR),
        "--output-dir", str(FINAL_MODEL_DIR / "final_eval"),
    )

FINAL_EVAL_METRICS = FINAL_MODEL_DIR / "final_eval" / "eval_metrics.json"
FINAL_EVAL_PREDICTIONS = FINAL_MODEL_DIR / "final_eval" / "eval_predictions.csv"
require_paths([FINAL_EVAL_METRICS, FINAL_EVAL_PREDICTIONS], "E7_FINAL evaluation")
final_eval_summary = json.loads(FINAL_EVAL_METRICS.read_text(encoding="utf-8"))
print(json.dumps({
    "n_eval": final_eval_summary.get("n_eval"),
    "n_features": final_eval_summary.get("n_features"),
    "threshold_independent_metrics": final_eval_summary.get("threshold_independent_metrics"),
    "official_min_tdcf": final_eval_summary.get("official_min_tdcf"),
    "fixed_threshold": final_eval_summary.get("fixed_threshold"),
}, indent=2))

## 12. Explainability: global and class-conditional SHAP

SHAP is run after the final model and evaluation outputs exist. The global script draws a seeded random sample of 2,000 evaluation recordings. The class-conditional script draws separate seeded samples of up to 1,000 bona fide and 1,000 spoof recordings. SHAP is computed in raw-margin/log-odds space for the bona-fide class. These are post-hoc model attributions, not causal acoustic explanations.

In [ ]:
if RUN_FINAL_SHAP:
    if SHAP_VERSION is None or MATPLOTLIB_VERSION is None:
        raise ImportError("SHAP and Matplotlib are required. Set INSTALL_PROJECT_EXTRAS=True and run the install cell.")
    run_tool(
        "tools/shap_global_e7.py",
        "--project-root", str(PROJECT_ROOT),
        "--feature-dir", str(FUSION_DIR),
        "--model-dir", str(FINAL_MODEL_DIR),
        "--output-dir", str(GLOBAL_SHAP_DIR),
        "--sample-size", SHAP_GLOBAL_SAMPLE_SIZE,
        "--batch-size", SHAP_BATCH_SIZE,
        "--seed", RANDOM_SEED,
        "--top-k", 25,
    )
    run_tool(
        "tools/shap_class_conditional_e7.py",
        "--project-root", str(PROJECT_ROOT),
        "--feature-dir", str(FUSION_DIR),
        "--model-dir", str(FINAL_MODEL_DIR),
        "--output-dir", str(CLASS_SHAP_DIR),
        "--samples-per-class", SHAP_CLASS_SAMPLES_PER_CLASS,
        "--batch-size", SHAP_BATCH_SIZE,
        "--seed", RANDOM_SEED,
        "--top-k", 25,
    )
else:
    print("Global and class-conditional SHAP skipped.")

## 13. Four individual SHAP explanations used in the paper

The saved case IDs correspond to the current E7_FINAL paper figures: false accept, false reject, correct accept and correct reject. If you change the model, training data, development threshold or evaluation set, re-check that these IDs still represent the intended case types before using the explanations in the paper.

In [ ]:
if RUN_FINAL_SHAP:
    for utterance_id, error_type in INDIVIDUAL_CASES:
        run_tool(
            "tools/explain_one_trial.py",
            "--feature-dir", str(FUSION_DIR),
            "--model-dir", str(FINAL_MODEL_DIR),
            "--prediction-file", str(FINAL_EVAL_PREDICTIONS),
            "--output-dir", str(INDIVIDUAL_SHAP_DIR),
            "--utterance-id", utterance_id,
            "--error-type", error_type,
            "--top-k", 20,
        )
else:
    print("Individual SHAP explanations skipped.")

## 14. Post-hoc E1–E7 condition analysis and paired statistics

This optional branch analyzes baseline/ablation E1–E7 evaluation predictions using the official manifest's environment and attack metadata. It reports false-accept rates for spoof trials and, in the paired analysis, compares the same utterance IDs across models using paired bootstrap intervals and exact McNemar tests with Holm correction. It does not feed metadata into XGBoost or into model selection.

Separately, `evaluate_tuned_e7.py` writes E7_FINAL post-hoc attack/environment FAR summaries into `eval_metrics.json`; these are consumed by the paper figure generator.

In [ ]:
if RUN_E1_E7_CONDITION_ANALYSIS:
    run_tool("tools/ASVspoof_PA_condition_analysis.py", "--root", str(PROJECT_ROOT))
else:
    print("E1–E7 condition analysis skipped.")

if RUN_PAIRED_CONDITION_STATISTICS:
    require_paths(
        [PROJECT_ROOT / "results" / "condition_analysis" / f"{exp}_joined_eval_predictions.csv" for exp in ABLATIONS],
        "E1–E7 condition-analysis inputs",
    )
    run_tool(
        "tools/paired_pa_condition_stats.py",
        "--input-dir", str(PROJECT_ROOT / "results" / "condition_analysis"),
        "--output-dir", str(PROJECT_ROOT / "results" / "paired_condition_statistics"),
        "--bootstrap", 5000,
        "--seed", 20261108,
    )
else:
    print("Paired condition statistics skipped.")

## 15. Generate the publication figures and tables

This uses the existing consolidated generator. Required inputs include the E1–E7 t-DCF comparison CSV, E7_FINAL evaluation outputs, global/class-conditional/individual SHAP outputs, and E7_FINAL post-hoc metadata summaries. It generates 15 PNG figures, 15 PDF figures, 21 CSV tables, and a ZIP package.

In [ ]:
if RUN_PAPER_FIGURES_AND_TABLES:
    require_paths([
        TDCF_DIR / "ablation_eer_tdcf_comparison.csv",
        FINAL_EVAL_METRICS,
        FINAL_EVAL_PREDICTIONS,
        GLOBAL_SHAP_DIR / "shap_global_group_importance.csv",
        CLASS_SHAP_DIR / "shap_class_conditional_group_importance.csv",
        *(INDIVIDUAL_SHAP_DIR / f"{uid}_group_contributions.csv" for uid, _ in INDIVIDUAL_CASES),
    ], "Paper-figure generation inputs")
    run_tool(
        "tools/generate_e7_paper_figures.py",
        "--project-root", str(PROJECT_ROOT),
        "--ablation-csv", str(TDCF_DIR / "ablation_eer_tdcf_comparison.csv"),
        "--final-model-dir", str(FINAL_MODEL_DIR),
        "--global-shap-dir", str(GLOBAL_SHAP_DIR),
        "--class-conditional-dir", str(CLASS_SHAP_DIR),
        "--individual-shap-dir", str(INDIVIDUAL_SHAP_DIR),
        "--output-dir", str(PAPER_FIGURE_DIR),
        "--top-k", 15,
    )
else:
    print("Paper figures/tables skipped.")

## 16. Final artifact audit and reproducibility metadata

This cell checks that the expected outputs exist and writes a compact run metadata file. It records software versions, run settings, feature dimensions and manifest counts. It intentionally does not embed the absolute dataset path, which is machine-specific.

In [ ]:
artifact_checks = {
    "manifests": [MANIFEST_DIR / f"PA_{s}.csv" for s in ("train", "dev", "eval")],
    "group_schemas": [GROUP_DIR / g / "feature_schema.json" for g in FEATURE_GROUPS],
    "fusion": [FUSION_DIR / "feature_schema.json", *[FUSION_DIR / f"X_{s}.npy" for s in ("train", "dev", "eval")]],
    "ablation": [ABLATION_DIR / exp / "metrics.json" for exp in ABLATIONS],
    "tuning": [TUNED_DIR / "search_manifest.json", TUNED_DIR / "trials.csv", FINAL_MODEL_DIR / "xgb_fusion.json"],
    "evaluation": [FINAL_EVAL_METRICS, FINAL_EVAL_PREDICTIONS],
    "global_shap": [GLOBAL_SHAP_DIR / "shap_global_group_importance.csv"],
    "class_shap": [CLASS_SHAP_DIR / "shap_class_conditional_group_importance.csv"],
    "paper_figures": [PAPER_FIGURE_DIR / "figures", PAPER_FIGURE_DIR / "tables"],
}
for category, paths in artifact_checks.items():
    missing = [str(p) for p in paths if not p.exists()]
    print(f"{category}:", "OK" if not missing else f"MISSING {missing}")

try:
    shap_version = shap.__version__
except Exception:
    shap_version = None

run_metadata = {
    "workflow": "ASVspoof 2019 PA complete reproducibility notebook",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__, "xgboost": xgb.__version__,
        "shap": shap_version, "matplotlib": MATPLOTLIB_VERSION,
    },
    "settings": {
        "seed": RANDOM_SEED,
        "extraction_workers": EXTRACTION_WORKERS,
        "xgb_n_jobs": XGB_N_JOBS,
        "xgb_device": XGB_DEVICE,
        "aggregation_stats": AGGREGATION_STATS,
        "ablation_n_estimators": ABLATION_N_ESTIMATORS,
        "ablation_early_stopping_rounds": ABLATION_EARLY_STOPPING,
        "tuning_trials": TUNING_TRIALS,
        "tuning_max_estimators": TUNING_MAX_ESTIMATORS,
        "tuning_early_stopping_rounds": TUNING_EARLY_STOPPING,
        "shap_global_sample_size": SHAP_GLOBAL_SAMPLE_SIZE,
        "shap_class_samples_per_class": SHAP_CLASS_SAMPLES_PER_CLASS,
    },
    "manifest_summary": manifest_summary,
    "feature_dimensions": EXPECTED_GROUP_WIDTHS,
    "fused_feature_count": 1350,
    "evaluation_policy": "E7_FINAL tuning reads train/dev only; official eval is evaluated after tuning with dev-selected operating threshold.",
}
metadata_dir = PROJECT_ROOT / "results" / "reproducibility"
metadata_dir.mkdir(parents=True, exist_ok=True)
(metadata_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Run metadata written:", metadata_dir / "run_metadata.json")

if PAPER_FIGURE_DIR.exists():
    figs = list((PAPER_FIGURE_DIR / "figures").glob("*.pdf")) if (PAPER_FIGURE_DIR / "figures").exists() else []
    pngs = list((PAPER_FIGURE_DIR / "figures").glob("*.png")) if (PAPER_FIGURE_DIR / "figures").exists() else []
    tables = list((PAPER_FIGURE_DIR / "tables").glob("*.csv")) if (PAPER_FIGURE_DIR / "tables").exists() else []
    print(f"Paper outputs: {len(pngs)} PNG, {len(figs)} PDF, {len(tables)} CSV")

## 17. What this notebook does not do

- It does not download or redistribute the ASVspoof corpus or official ASV score files. Obtain them from the official corpus distribution and point `DATASET_ROOT` / `ASV_SCORE_DIR` to the local copies.
- It does not run MATLAB reference code or recreate the published papers' experiments. It runs this project's Python implementation.
- It does not automatically copy the paper PDF or change the LaTeX manuscript.
- It does not guarantee identical runtimes or bitwise-identical GPU training across different hardware, driver and library versions. Save the generated `results/reproducibility/run_metadata.json`, package versions, and tuning manifest with the research release.
- A fresh run requires all source modules under `src/`, the tool scripts, the ASVspoof 2019 PA dataset/protocol files, and official ASV score files. Feature matrices and caches can be regenerated by this notebook.

### Resume guidance

If a stage fails, fix the reported error and rerun that cell. Feature extraction uses configuration-fingerprinted caches and will skip valid existing utterances. Group matrices and ablation models are skipped when the corresponding output files already exist and the skip switches are enabled. The E7 tuning script resumes from completed trials when its `search_manifest.json` matches the requested settings. If you intentionally change feature parameters, rebuild the corresponding group matrices and fusion matrices, and use a new tuning output directory or explicitly start a fresh tuning search.